In [ ]:
import tkinter as tk
from tkinter import messagebox
import random
import time

# A simple representation of a single cell on the Minesweeper board.
class Cell:
    def __init__(self, row, col):
        self.row = row
        self.col = col
        self.is_mine = False      # whether this cell contains a mine
        self.revealed = False     # whether the player has revealed this cell
        self.flagged = False      # whether the player has placed a flag here
        self.adjacent = 0         # number of mines adjacent to this cell

# The main game logic and GUI for Minesweeper
class Minesweeper:
    def __init__(self, master, rows=9, cols=9, mines=10):
        # store parameters and initialize state
        self.master = master
        self.rows = rows
        self.cols = cols
        self.mines = mines
        self.first_click = True   # track whether the first click has occured
        self.start_time = None
        self.elapsed = 0
        self.timer_running = False

        # configure window title
        self.master.title('Minesweeper')

        # Top control frame with size/mines settings and labels
        top = tk.Frame(master)
        top.pack(padx=5, pady=5, anchor='n')

        tk.Label(top, text='Rows:').grid(row=0, column=0)
        self.rows_var = tk.IntVar(value=self.rows)
        tk.Spinbox(top, from_=5, to=30, width=4, textvariable=self.rows_var, command=self._update_size).grid(row=0, column=1)

        tk.Label(top, text='Cols:').grid(row=0, column=2)
        self.cols_var = tk.IntVar(value=self.cols)
        tk.Spinbox(top, from_=5, to=30, width=4, textvariable=self.cols_var, command=self._update_size).grid(row=0, column=3)

        tk.Label(top, text='Mines:').grid(row=0, column=4)
        self.mines_var = tk.IntVar(value=self.mines)
        tk.Spinbox(top, from_=1, to=200, width=6, textvariable=self.mines_var, command=self._update_size).grid(row=0, column=5)

        self.mine_label = tk.Label(top, text=f'Mines: {self.mines}')
        self.mine_label.grid(row=0, column=6, padx=10)

        self.time_label = tk.Label(top, text='Time: 0')
        self.time_label.grid(row=0, column=7, padx=10)

        tk.Button(top, text='Restart', command=self.restart).grid(row=0, column=8, padx=5)

        # create frame to hold the board buttons
        self.board_frame = tk.Frame(master)
        self.board_frame.pack(padx=5, pady=5)

        # initial board creation
        self._create_board()

    def _update_size(self):
        # Adjust board size when spinboxes change and restart game.
        try:
            r = int(self.rows_var.get())
            c = int(self.cols_var.get())
            m = int(self.mines_var.get())
        except Exception:
            return
        self.rows = max(5, min(30, r))
        self.cols = max(5, min(30, c))
        max_mines = self.rows * self.cols - 1
        m = max(1, min(max_mines, m))
        self.mines = m
        self.mine_label.config(text=f'Mines: {self.mines}')
        self.restart()

    def _create_board(self):
        # create the cell model grid and corresponding button widgets
        self.cells = [[Cell(r, c) for c in range(self.cols)] for r in range(self.rows)]
        self.buttons = [[None for _ in range(self.cols)] for _ in range(self.rows)]

        for r in range(self.rows):
            for c in range(self.cols):
                b = tk.Button(self.board_frame, width=2, height=1, font=('TkDefaultFont', 12, 'bold'))
                b.grid(row=r, column=c)
                # bind left and right click handlers with current coordinates
                b.bind('<Button-1>', lambda e, rr=r, cc=c: self.on_left_click(rr, cc))
                b.bind('<Button-3>', lambda e, rr=r, cc=c: self.on_right_click(rr, cc))
                self.buttons[r][c] = b

        # adjust window minimum size to fit the board
        self.master.update_idletasks()
        w = self.board_frame.winfo_width() + 20
        h = self.board_frame.winfo_height() + 80
        self.master.minsize(w, h)

    def place_mines(self, safe_r, safe_c):
        # randomly place mines, avoiding the first-clicked cell and its neighbors
        positions = [(r, c) for r in range(self.rows) for c in range(self.cols)]
        safe = set()
        for rr in range(safe_r - 1, safe_r + 2):
            for cc in range(safe_c - 1, safe_c + 2):
                if 0 <= rr < self.rows and 0 <= cc < self.cols:
                    safe.add((rr, cc))
        choices = [p for p in positions if p not in safe]
        mines_to_place = min(self.mines, len(choices))
        mines = random.sample(choices, mines_to_place)
        for (r, c) in mines:
            self.cells[r][c].is_mine = True

        # compute adjacent mine counts for non-mine cells
        for r in range(self.rows):
            for c in range(self.cols):
                if self.cells[r][c].is_mine:
                    continue
                count = 0
                for rr in range(r - 1, r + 2):
                    for cc in range(c - 1, c + 2):
                        if 0 <= rr < self.rows and 0 <= cc < self.cols and self.cells[rr][cc].is_mine:
                            count += 1
                self.cells[r][c].adjacent = count

    def on_left_click(self, r, c):
        # handle left-click events: start timer, place mines on first click,
        # reveal cells, and check for win/loss conditions
        if not self.timer_running:
            self.start_timer()
        cell = self.cells[r][c]
        if self.first_click:
            self.place_mines(r, c)
            self.first_click = False
        if cell.flagged or cell.revealed:
            return
        if cell.is_mine:
            self.reveal_mine(r, c)
            self.game_over(False)
            return
        self.reveal_cell(r, c)
        if self.check_win():
            self.game_over(True)

    def on_right_click(self, r, c):
        # toggle flag on right-click if cell is not yet revealed
        cell = self.cells[r][c]
        if cell.revealed:
            return
        cell.flagged = not cell.flagged
        b = self.buttons[r][c]
        if cell.flagged:
            b.config(text='⚑')
        else:
            b.config(text='')
        self.update_mine_count()

    def reveal_cell(self, r, c):
        # show the content of a cell, flood fill if zero adjacent mines
        cell = self.cells[r][c]
        b = self.buttons[r][c]
        if cell.revealed or cell.flagged:
            return
        cell.revealed = True
        b.config(relief=tk.SUNKEN)
        if cell.adjacent > 0:
            b.config(text=str(cell.adjacent), disabledforeground=self._color_for_number(cell.adjacent))
        else:
            b.config(text='')
        b.config(state=tk.DISABLED)

        if cell.adjacent == 0:
            # recursively reveal neighbors
            for rr in range(r - 1, r + 2):
                for cc in range(c - 1, c + 2):
                    if 0 <= rr < self.rows and 0 <= cc < self.cols:
                        if not self.cells[rr][cc].revealed:
                            self.reveal_cell(rr, cc)

    def reveal_mine(self, exploded_r, exploded_c):
        # show all mines and mark the one that was exploded
        for r in range(self.rows):
            for c in range(self.cols):
                if self.cells[r][c].is_mine:
                    b = self.buttons[r][c]
                    b.config(text='*', disabledforeground='black')
                    b.config(state=tk.DISABLED)
        be = self.buttons[exploded_r][exploded_c]
        be.config(text='💥')

    def check_win(self):
        # win condition: all non-mine cells are revealed
        for r in range(self.rows):
            for c in range(self.cols):
                cell = self.cells[r][c]
                if not cell.is_mine and not cell.revealed:
                    return False
        return True

    def game_over(self, won):
        # stop timer and display result message, disable further interaction
        self.stop_timer()
        if won:
            messagebox.showinfo('You win!', f'Congratulations — you cleared the board in {self.elapsed:.0f} seconds!')
            # show flags on mines
            for r in range(self.rows):
                for c in range(self.cols):
                    if self.cells[r][c].is_mine:
                        self.buttons[r][c].config(text='⚑')
        else:
            messagebox.showinfo('Game over', 'Boom! You clicked on a mine.')
        # unbind events from all buttons to prevent further clicks
        for r in range(self.rows):
            for c in range(self.cols):
                try:
                    self.buttons[r][c].unbind('<Button-1>')
                    self.buttons[r][c].unbind('<Button-3>')
                except Exception:
                    pass

    def update_mine_count(self):
        # update mines remaining label based on flags placed
        flags = sum(1 for r in range(self.rows) for c in range(self.cols) if self.cells[r][c].flagged)
        remaining = max(0, self.mines - flags)
        self.mine_label.config(text=f'Mines: {remaining}')

    def start_timer(self):
        # begin game timer
        self.start_time = time.time()
        self.timer_running = True
        self._tick()

    def _tick(self):
        # update timer label every quarter second
        if not self.timer_running:
            return
        self.elapsed = time.time() - self.start_time
        self.time_label.config(text=f'Time: {int(self.elapsed)}')
        self.master.after(250, self._tick)

    def stop_timer(self):
        # stop the game timer
        self.timer_running = False

    def restart(self):
        # reset game state and rebuild the board
        for widget in self.board_frame.winfo_children():
            widget.destroy()
        self.first_click = True
        self.start_time = None
        self.elapsed = 0
        self.timer_running = False
        self.rows = int(self.rows_var.get())
        self.cols = int(self.cols_var.get())
        self.mines = int(self.mines_var.get())
        max_mines = max(1, self.rows * self.cols - 1)
        if self.mines > max_mines:
            self.mines = max_mines
            self.mines_var.set(self.mines)
        self.mine_label.config(text=f'Mines: {self.mines}')
        self.time_label.config(text='Time: 0')
        self._create_board()

    def _color_for_number(self, n):
        # helper to choose text color based on adjacent mine count
        colors = {
            1: 'blue', 2: 'green', 3: 'red', 4: 'darkblue',
            5: 'darkred', 6: 'turquoise', 7: 'black', 8: 'gray'
        }
        return colors.get(n, 'black')

# entry point when script is run directly
if __name__ == '__main__':
    root = tk.Tk()
    app = Minesweeper(root, rows=9, cols=9, mines=10)
    root.mainloop()